# Stage 09 structure-aware redesign on SageMaker

This notebook runs the new Stage 09 pipeline:
1. define a stricter edit space,
2. build the structural-surrogate dataset,
3. train/configure the surrogate,
4. run structure-aware localized search,
5. prefilter candidates,
6. validate the top panel with the fixed Stage 08 structural validator,
7. build the final Stage 09 comparison report.


In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path.cwd()
print('Working directory:', ROOT)
assert (ROOT / 'pyproject.toml').exists(), 'Run this notebook from the repo root in SageMaker JupyterLab.'
print('Repo ready.')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Environment ready.')


## Configure paths

Adjust the benchmark / Stage 07 / Stage 08 paths below if your repo layout differs.


In [ ]:
STAGE07_CONTEXT = 'results/stage07/context/stage07_context.base.json'
STRICT_CSV = 'data/processed/rbp_dataset_eskapee_strict.csv'
STAGE07_RANKED = 'results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv'
BASELINE_STAGE08 = 'results/stage08/structural_fasttrack_top3/stage08_structural_fasttrack_summary.csv'
PREDICTOR_MODEL = 'results/broad/linear_probe/seed_42/model.joblib'
LABEL_CLASSES = 'results/broad/linear_probe/seed_42/label_classes.json'
OUT_ROOT = 'results/stage09'

for path in [STAGE07_CONTEXT, STRICT_CSV, STAGE07_RANKED, PREDICTOR_MODEL, LABEL_CLASSES]:
    assert Path(path).exists(), f'Missing required path: {path}'

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print('Output root:', OUT_ROOT)


## Step 1 — define a stricter edit space


In [ ]:
!python scripts/09a_define_edit_space.py \
  --context_json {STAGE07_CONTEXT} \
  --strict_csv {STRICT_CSV} \
  --output_json {OUT_ROOT}/edit_space/stage09_edit_space.json \
  --max_edit_positions 12 \
  --soft_buffer_positions 6 \
  --min_mutations 3 \
  --max_mutations 8


## Step 2 — build and configure the structural surrogate

Use the baseline Stage 08 structural summary as the initial supervision source.


In [ ]:
!python scripts/09b_build_structure_surrogate_dataset.py \
  --context_json {STAGE07_CONTEXT} \
  --ranked_csv {STAGE07_RANKED} \
  --structural_csv {BASELINE_STAGE08} \
  --out_csv {OUT_ROOT}/surrogate/stage09_surrogate_dataset.csv \
  --out_json {OUT_ROOT}/surrogate/stage09_surrogate_summary.json

!python scripts/09c_train_structure_surrogate.py \
  --dataset_csv {OUT_ROOT}/surrogate/stage09_surrogate_dataset.csv \
  --summary_json {OUT_ROOT}/surrogate/stage09_surrogate_summary.json \
  --out_model {OUT_ROOT}/surrogate/stage09_surrogate.joblib


## Step 3 — run structure-aware localized search


In [ ]:
!python scripts/09d_localized_search.py \
  --context_json {STAGE07_CONTEXT} \
  --edit_space_json {OUT_ROOT}/edit_space/stage09_edit_space.json \
  --strict_csv {STRICT_CSV} \
  --predictor_model {PREDICTOR_MODEL} \
  --label_classes_json {LABEL_CLASSES} \
  --surrogate_model {OUT_ROOT}/surrogate/stage09_surrogate.joblib \
  --out_csv {OUT_ROOT}/search/stage09_search_candidates.csv \
  --out_json {OUT_ROOT}/search/stage09_search_run.json \
  --esm_model facebook/esm2_t12_35M_UR50D \
  --batch_size 4 \
  --rounds 4 \
  --beam_width 24 \
  --proposals_per_parent 18 \
  --max_mutations 8


## Step 4 — prefilter candidates before expensive structural validation


In [ ]:
!python scripts/09e_structural_prefilter.py \
  --search_csv {OUT_ROOT}/search/stage09_search_candidates.csv \
  --search_meta_json {OUT_ROOT}/search/stage09_search_run.json \
  --out_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --top_k 12 \
  --max_structural_risk 0.55 \
  --min_predicted_plddt 55 \
  --max_predicted_rmsd 4.5 \
  --min_sequence_identity 0.93


## Step 5 — validate top 10 and top 3 with the fixed Stage 08 validator


In [ ]:
!python scripts/09f_validate_stage09_candidates.py \
  --prefilter_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --context_json {STAGE07_CONTEXT} \
  --out_dir {OUT_ROOT}/validation_top10 \
  --top_k 10 \
  --device cuda \
  --chunk_size 128 \
  --num_recycles 1 \
  --resume

!python scripts/09f_validate_stage09_candidates.py \
  --prefilter_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --context_json {STAGE07_CONTEXT} \
  --out_dir {OUT_ROOT}/validation_top3 \
  --top_k 3 \
  --device cuda \
  --chunk_size 128 \
  --num_recycles 1 \
  --resume


## Step 6 — build the final Stage 09 report


In [ ]:
!python scripts/09g_make_stage09_report.py \
  --search_csv {OUT_ROOT}/search/stage09_search_candidates.csv \
  --prefilter_csv {OUT_ROOT}/prefilter/stage09_prefilter_top12.csv \
  --validation_csv {OUT_ROOT}/validation_top10/stage08_structural_fasttrack_summary.csv \
  --baseline_stage08_csv {BASELINE_STAGE08} \
  --out_dir {OUT_ROOT}/report


## Step 7 — inspect outputs


In [ ]:
import pandas as pd
from pathlib import Path

display(pd.read_csv(f'{OUT_ROOT}/prefilter/stage09_prefilter_top12.csv').head(12))
display(pd.read_csv(f'{OUT_ROOT}/validation_top10/stage08_structural_fasttrack_summary.csv'))
print(Path(f'{OUT_ROOT}/report/stage09_report.md').read_text())
